Module 01: Exploratory Data Analysis for Demand & Inventory with Polars

This notebook performs exploratory data analysis (EDA) for Module 01 of the **"Intelligent System for Supply Chain Management"** project.  

The primary goal is to optimize inventory and purchasing management, with a target of **reducing overstocking by 20%** within six months.

---

## Import Libraries

In [1]:
import polars as pl
import polars.selectors as cs
import json
import plotly.express as px
import plotly.io as pio
from pathlib import Path
from polars_info import print_df_info
from datetime import date

# Install dependencies as needed:
# pip install kagglehub[polars-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

import warnings
warnings.filterwarnings('ignore')

# Set up display options and plotting template
pio.templates.default = "plotly_white"
px.defaults.width = 800
px.defaults.height = 600

## Load Dataset

In [2]:
# Paths
path_docs = Path("..") / "docs"
path_processed = Path("..") / "data" / "processed"

In [3]:
# Set the path to the file you'd like to load
file_path = "1M_grocery_data_pl.parquet"

# Load the latest version
lf = kagglehub.dataset_load(
  KaggleDatasetAdapter.POLARS,
  "robertobalbinotti/synthetic-grocery-data",
  file_path,
  # Provide any additional arguments like
  # sql_query, polars_frame_type, or 
  # polars_kwargs.
  # See the documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpolars
)

In [4]:
# Load column descriptions from JSON file into a dictionary for reference or documentation
with open(path_docs / 'column_descriptions_polars.json') as f:
    column_descriptions = json.load(f)

# Data Cleaning and Preprocessing

In [5]:
print_df_info(lf.collect())

<class 'polars.dataframe.frame.DataFrame'>
Shape: (1,000,658, 31)
Estimated size: 182.25 MiB
Columns:
  #  Column                        Dtype         Non-Null    Null   Null%
  0  order_purchase_date           Date          1,000,658       0   0.00%
  1  received_date                 Date          1,000,658       0   0.00%
  2  product_id                    String        1,000,658       0   0.00%
  3  product                       String        1,000,658       0   0.00%
  4  category                      String        1,000,658       0   0.00%
  5  sub_category                  String        1,000,658       0   0.00%
  6  sales_demand                  String        1,000,658       0   0.00%
  7  sales_volume                  UInt16        1,000,658       0   0.00%
  8  seasonality                   List(String)  1,000,658       0   0.00%
  9  storage_recommendation        String        1,000,658       0   0.00%
 10  unit_of_measurement           String        1,000,658       0   0.00%

DFInfoSummary(rows=1000658, cols=31, estimated_size_bytes=191102740, dtypes={'order_purchase_date': Date, 'received_date': Date, 'product_id': String, 'product': String, 'category': String, 'sub_category': String, 'sales_demand': String, 'sales_volume': UInt16, 'seasonality': List(String), 'storage_recommendation': String, 'unit_of_measurement': String, 'shelf_life_days': UInt16, 'maximum_days_on_sale': UInt16, 'supplier_id': String, 'supplier': String, 'supplier_rating': UInt8, 'distance_km': UInt16, 'moq': UInt16, 'delivery_days': Float16, 'transit_time': Float16, 'in_season': Boolean, 'is_holiday': Boolean, 'day_classification': String, 'is_weekend': Boolean, 'min_stock': UInt16, 'max_stock': UInt16, 'stock_quantity': UInt16, 'temperature_classification': String, 'precipitation_classification': String, 'wind_classification': String, 'weather_severity': String})

In [63]:
df = lf.with_columns(
    pl.col(pl.Utf8).cast(pl.Categorical()),
)

df.show(2)

order_purchase_date,received_date,product_id,product,category,sub_category,sales_demand,sales_volume,seasonality,storage_recommendation,unit_of_measurement,shelf_life_days,maximum_days_on_sale,supplier_id,supplier,supplier_rating,distance_km,moq,delivery_days,transit_time,in_season,is_holiday,day_classification,is_weekend,min_stock,max_stock,stock_quantity,temperature_classification,precipitation_classification,wind_classification,weather_severity
date,date,cat,cat,cat,cat,cat,u16,list[str],cat,cat,u16,u16,cat,cat,u8,u16,u16,f16,f16,bool,bool,cat,bool,u16,u16,u16,cat,cat,cat,cat
2022-12-07,2022-12-09,"""1169187|P""","""Tomato""","""Fresh Foods""","""Vegetables""","""High""",164,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""",7,3,"""1194877|S""","""ValleyFresh Farms""",4,85,100,0.605469,2.4140625,false,false,"""Weekday""",false,269,369,292,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-06,2022-12-09,"""1741974|P""","""Mozzarella Cheese""","""Dairy & Alternatives""","""Dairy""","""High""",108,[],"""Refrigerated""","""lb""",14,5,"""1422853|S""","""Artisan Cheesemakers""",5,95,40,0.931641,2.769531,false,false,"""Weekday""",false,237,277,275,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""


In [27]:
received_date = df.select("received_date").collect().to_series().sort().unique()

expected_date_range = pl.date_range(
    start= received_date.min(),
    end= received_date.max(),
    interval="1d",
    eager=True
)

is_complete = received_date.len() == expected_date_range.len()
missing_dates = expected_date_range.filter(~expected_date_range.is_in(received_date))

print("Complete Received Date Range?\n", is_complete)

Complete Received Date Range?
 True


# Feature Engineering

In [114]:
df = df.with_columns(
    (pl.col("received_date") - pl.col("order_purchase_date")).alias("delivery_lag")
).with_columns(
    pl.when(pl.col("delivery_lag") > pl.duration(days=pl.col("shelf_life_days")))
    .then(pl.lit("Expired"))
    .when(pl.col("delivery_lag") > pl.duration(days=pl.col("maximum_days_on_sale")))
    .then(pl.lit("Nearing"))
    .otherwise(pl.lit("Safe"))
    .cast(pl.Categorical())
    .alias("expiration_status")
).with_columns(
    (pl.col("order_purchase_date").dt.year())
    .cast(pl.UInt16).alias("year"),
    (pl.col("stock_quantity") - pl.col("sales_volume"))
    .cast(pl.Float32)
    .alias("closing_stock")
).with_columns(
    pl.col("sales_volume").sum().over(["product", "year"])
    .cast(pl.Float32)
    .alias("total_sales"),
    pl.col("closing_stock").mean().over(["product", "year"])
    .cast(pl.Float32)
    .alias("average_stock"), 
).with_columns(
    pl.when(pl.col("average_stock") > 0)
    .then(pl.col("total_sales") / pl.col("average_stock"))
    .otherwise(0.0)
    .cast(pl.Float32)
    .alias("inventory_turnover_rate")
).with_columns(
    pl.col("inventory_turnover_rate").mean().over(["product"])
    .cast(pl.Float32)
    .alias("average_turnover_rate")
).with_columns(
    (pl.col("received_date").max() - pl.col("received_date").min())
    .dt.total_days()
    .alias("period_days")
).with_columns(
    (pl.col("period_days") / pl.col("inventory_turnover_rate"))
    .floor()
    .cast(pl.UInt16)
    .alias("doi_inventory_turnover")
).drop(pl.col("period_days"))

df.show(2)

order_purchase_date,received_date,product_id,product,category,sub_category,sales_demand,sales_volume,seasonality,storage_recommendation,unit_of_measurement,shelf_life_days,maximum_days_on_sale,supplier_id,supplier,supplier_rating,distance_km,moq,delivery_days,transit_time,in_season,is_holiday,day_classification,is_weekend,min_stock,max_stock,stock_quantity,temperature_classification,precipitation_classification,wind_classification,weather_severity,delivery_lag,expiration_status,year,closing_stock,total_sales,average_stock,inventory_turnover_rate,average_turnover_rate,doi_inventory_turnover
date,date,cat,cat,cat,cat,cat,u16,list[str],cat,cat,u16,u16,cat,cat,u8,u16,u16,f16,f16,bool,bool,cat,bool,u16,u16,u16,cat,cat,cat,cat,duration[μs],cat,u16,f32,f32,f32,f32,f32,u16
2022-12-07,2022-12-09,"""1169187|P""","""Tomato""","""Fresh Foods""","""Vegetables""","""High""",164,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""",7,3,"""1194877|S""","""ValleyFresh Farms""",4,85,100,0.605469,2.4140625,false,false,"""Weekday""",false,269,369,292,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""",2d,"""Safe""",2022,128.0,38744.0,5642.839355,6.866047,67.31739,148
2022-12-06,2022-12-09,"""1741974|P""","""Mozzarella Cheese""","""Dairy & Alternatives""","""Dairy""","""High""",108,[],"""Refrigerated""","""lb""",14,5,"""1422853|S""","""Artisan Cheesemakers""",5,95,40,0.931641,2.769531,false,false,"""Weekday""",false,237,277,275,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""",3d,"""Safe""",2022,167.0,12784.0,6806.277344,1.878266,23.328737,541


In [115]:
## Add a description for the 'inventory_turnover_rate' column
column_descriptions.update({
    "year": "The specific calendar or fiscal year for which the inventory data is recorded.",
    "closing_stock": "The total value or quantity of remaining inventory available at the end of the reporting period.",
    "total_sales": "The total revenue or total units sold over the specified accounting period.",
    "average_stock": "The mean value or quantity of inventory held during a period, calculated as (Beginning Stock + Ending Stock) / 2.",
    "inventory_turnover_rate": "A ratio showing how many times a company's inventory is sold and replaced over a given period, calculated as Cost of Goods Sold (COGS) divided by Average Inventory.",
    "average_turnover_rate": "The mean rate of inventory turnover calculated across multiple periods or product categories.",
    "doi_inventory_turnover": "Stock coverage in days."
})

# Exploratory Data Analysis (EDA)